<a href="https://colab.research.google.com/github/Vga18/diplomado/blob/main/Proyecto_diplomado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cont-AI**

## **Planteamiento del problema**

El desarrollo de pólizas es un proceso automático y tardado, ya que requiere tener un criterio para decidir que gastos van a que cuenta contable, y despues de esta clasificación se genera la poliza correspondiente.

Por ello este sistema tienen como objetivo agilizar la capturación de pólizas de egresos mediante la facturas recibidas. Con esto se acota el proyecto a solo las facturas XML-CFDI recibidas, y al catalogo de cuentas contables que maneja el Sistema Aspel COI versión 9

Por lo tanto el obejtivo de este notebook es genera un modelo que clasifique las facturas a su debida cuenta contable

## Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


pd.set_option("display.max_columns",200)
pd.set_option("display.max_rows",100)

## Funciones Auxiliares

In [ ]:
import re

def extract_invoice_info(concept_po):
    """
    Extraer el número de factura y el nombre de la empresa de la columna 'CONCEP_PO'
    del DataFrame `pol_cat` utilizando expresiones regulares y división de cadenas,
    manejando casos de valores faltantes.
    """
    invoice_number = None
    company_name = None

    if pd.isna(concept_po):
        return invoice_number, company_name

    # Convert to string to avoid errors with non-string types
    concept_po_str = str(concept_po)

    # Extract invoice number using regex
    # Looks for 'NO.' or 'FACTURA NO.' followed by alphanumeric characters and hyphens
    invoice_match = re.search(r'(?:NO\.|FACTURA NO\.)\s*([A-Za-z0-9\-]+)', concept_po_str, re.IGNORECASE)
    if invoice_match:
        invoice_number = invoice_match.group(1)

    # Extract company name by splitting at the first comma
    parts = concept_po_str.split(',', 1)
    if len(parts) > 1:
        company_name = parts[1].strip()

    return invoice_number, company_name

## Datos

Para la realización de este proyecto se cuenta con las siguientes fuentes de datos:

In [ ]:
catalogo = pd.read_csv(r"/content/Datos/catalogo de cuentas contables.csv")
polizas22 = pd.read_csv(r"/content/Datos/polizas22.csv")
polizas23 = pd.read_csv(r"/content/Datos/polizas23.csv")
polizas24 = pd.read_csv(r"/content/Datos/polizas24.csv")
polizas25 = pd.read_csv(r"/content/Datos/polizas25.csv")
polizas26 = pd.read_csv(r"/content/Datos/polizas26.csv")
# xmls =  pd.read_csv(r"/content/Datos/xmls.csv")

Que mediante el dataframe 'catalogo' toda la información relacionada las cuentas contables, los dataframe del que tiene como subnombre 'polizas' es del como fue capturado la poliza (que mediante el nombre de la poliza se registro los datos de la factura que hace relación a esta), Y con el dataframe 'xmls' son todos los xmls recibidos

Con todo esto es posible, contruir el dataframe muestral que nos ayudara para la creeación del modelo.

# TAD

In [ ]:
catalogo.head(3)

,NUM_CTA,STATUS,TIPO,NOMBRE,DEPTSINO,BANDMULTI,BANDAJT,CTA_PAPA,CTA_RAIZ,NIVEL,CTA_COMP,NATURALEZA,RFC,CODAGRUP,CAPTURACHEQUE,CAPTURAUUID,BANCO,CTABANCARIA,CAPCHEQTIPOMOV,NOINCLUIRXML,IDFISCAL,ESFLUJODEEFECTIVO,BANCOEXTRANJERO,RFCFLUJO
305,117000300300000000003,A,D,VICENTE VEGA RAMIREZ,N,1,N,117000300000000000002,117000000000000000001,3,NaN,0,NaN,NaN,0,0,0,NaN,N,0,NaN,NaN,NaN,NaN
306,212000100200000000003,A,D,FRANCISCO D. VEGA HERNANDEZ,N,1,N,212000100000000000002,212000000000000000001,3,NaN,1,NaN,NaN,0,0,0,NaN,N,0,NaN,NaN,NaN,NaN
307,111000200000000000002,A,D,TAG CUENTA.0002706454,N,1,N,111000000000000000001,111000000000000000001,2,0.0,0,NaN,NaN,0,0,0,NaN,N,0,NaN,NaN,NaN,NaN


Solo nos intera ciertos datos

In [ ]:
catalogo = catalogo[["NUM_CTA","STATUS","TIPO","NOMBRE","NIVEL","NATURALEZA"]]
catalogo = catalogo[catalogo['NIVEL'] != 1] # Ya que en las cuentas contables, se manejan por jerarquias teneiendo como nivel 1 las que son cuentas raiz
catalogo.sample(3)

,NUM_CTA,STATUS,TIPO,NOMBRE,NIVEL,NATURALEZA
107,321000100000000000002,A,D,RESERVA PARA REINVERSION,2,1
302,117000300200000000003,A,D,ESTEBAN GAMALIEL RAMIREZ HERNANDEZ,3,0
62,211000100000000000002,A,D,PROVEEDOR #,2,1
242,640000100000000000002,A,D,AMORTIZACION DE PATENTES Y MARCAS,2,0
276,910001200000000000002,A,D,PÉRDIDAS FISCALES POR AMORTIZAR,2,0


In [ ]:
polizas26.head(3)

,TIPO_POLI,NUM_POLIZ,NUM_PART,PERIODO,EJERCICIO,NUM_CTA,FECHA_POL,CONCEP_PO,DEBE_HABER,MONTOMOV,NUMDEPTO,TIPCAMBIO,CONTRAPAR,ORDEN,CCOSTOS,CGRUPOS,IDINFADIPAR,IDUUID
0,Eg,1,1.0,1,2026,610004200000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-6315210, OPERADORA CONCESION...",D,994.83,0,1.0,0,1,0,0,-1,-1
1,Eg,1,2.0,1,2026,120000100000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-6315210, OPERADORA CONCESION...",D,159.17,0,1.0,0,2,0,0,-1,-1
2,Eg,1,3.0,1,2026,111000200000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-6315210, OPERADORA CONCESION...",H,1154.00,0,1.0,0,3,0,0,-1,-1
3,Eg,2,1.0,1,2026,610004200000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-1760426, CONCESIONARIA MEXIQ...",D,200.00,0,1.0,0,1,0,0,-1,-1
4,Eg,2,2.0,1,2026,120000100000000000002,2026-01-01 00:00:00.000,"PAGO FACTURA NO.T-1760426, CONCESIONARIA MEXIQ...",D,32.00,0,1.0,0,2,0,0,-1,-1


Tambien aqui nos interesan ciertas variables

In [ ]:
variables_destables = ["TIPO_POLI","PERIODO","EJERCICIO","NUM_CTA","FECHA_POL","CONCEP_PO","DEBE_HABER"]

Polizas22 = polizas22[variables_destables]
Polizas23 = polizas23[variables_destables]
Polizas24 = polizas24[variables_destables]
Polizas25 = polizas25[variables_destables]
polizas26 = polizas26[variables_destables]

polizas = pd.concat([Polizas22,Polizas23,Polizas24,Polizas25,polizas26])

In [ ]:
polizas = polizas[polizas["DEBE_HABER"] == "D"]
polizas = polizas[polizas["CONCEP_PO"] != "CANCELADO"]

In [ ]:
polizas.isnull().sum().sort_values(ascending=False)

,0
CONCEP_PO,4
PERIODO,0
TIPO_POLI,0
EJERCICIO,0
NUM_CTA,0
FECHA_POL,0
DEBE_HABER,0


In [ ]:
polizas = polizas.dropna() #Son polizas que no se tienen infomración

In [ ]:
polizas = polizas[polizas['CONCEP_PO'].str.contains("FACTURA")]

In [ ]:
polizas.sample(3)

,TIPO_POLI,PERIODO,EJERCICIO,NUM_CTA,FECHA_POL,CONCEP_PO,DEBE_HABER
5,Dr,2,2022,610003900000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D
6,Dr,2,2022,120000100000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D
10,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D
11,Eg,2,2022,120000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D
14,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. DFA424799, CT INTERNACI...",D
...,...,...,...,...,...,...,...
144,Eg,1,2026,112000100100000000003,2026-01-28 00:00:00.000,"PAGO FACTURA NO., AUTOZONE",D
145,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D
146,Eg,1,2026,120000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D
149,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.GCM-049355, GRUPO DAISTYTEK",D


In [ ]:
pol_cat = pd.merge(polizas,catalogo,how="left",left_on="NUM_CTA",right_on="NUM_CTA")

In [ ]:
pol_cat

,TIPO_POLI,PERIODO,EJERCICIO,NUM_CTA,FECHA_POL,CONCEP_PO,DEBE_HABER,STATUS,TIPO,NOMBRE,NIVEL,NATURALEZA
0,Dr,2,2022,610003900000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D,A,D,TELEFONOS,2,0
1,Dr,2,2022,120000100000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D,A,D,IVA ACREDITABLE PAGADO,2,0
2,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D,A,D,PROVEEDOR #,2,1
3,Eg,2,2022,120000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D,A,D,IVA ACREDITABLE PAGADO,2,0
4,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. DFA424799, CT INTERNACI...",D,A,D,PROVEEDOR #,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
4741,Eg,1,2026,112000100100000000003,2026-01-28 00:00:00.000,"PAGO FACTURA NO., AUTOZONE",D,A,D,CTA: 0118116830,3,0
4742,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D,A,D,PROVEEDOR #,2,1
4743,Eg,1,2026,120000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D,A,D,IVA ACREDITABLE PAGADO,2,0
4744,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.GCM-049355, GRUPO DAISTYTEK",D,A,D,PROVEEDOR #,2,1


In [ ]:
pol_cat = pol_cat[~pol_cat["NUM_CTA"].astype(str).str.startswith("1")].reset_index(drop=True)

Ya que queremos extraer la factura de la variable 'CONCEP_PO' para depues unirlos con el dataframe 'xmls', se hara uso de la función auxiliar

```
def extract_invoice_info(concept_po)
```

In [ ]:
# Apply the function to create new columns
pol_cat[['Factura', 'Empresa']] = pol_cat['CONCEP_PO'].apply(lambda x: pd.Series(extract_invoice_info(x)))

,TIPO_POLI,PERIODO,EJERCICIO,NUM_CTA,FECHA_POL,CONCEP_PO,DEBE_HABER,STATUS,TIPO,NOMBRE,NIVEL,NATURALEZA,Factura,Empresa
0,Dr,2,2022,610003900000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D,A,D,TELEFONOS,2,0,B1190023669T10,None
1,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D,A,D,PROVEEDOR #,2,1,FDF3684614,INGRAM MICRO SA DE CV
2,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. DFA424799, CT INTERNACI...",D,A,D,PROVEEDOR #,2,1,DFA424799,CT INTERNACIONAL DEL NOROESTE SA DE CV
3,Eg,2,2022,211000100000000000002,2022-02-08 00:00:00.000,"PAGO DE LA FACTURA NO. DFA425973, CT INTERNACI...",D,A,D,PROVEEDOR #,2,1,DFA425973,CT INTERNACIONAL DEL NOROESTE SA DE CV
4,Eg,2,2022,211000100000000000002,2022-02-10 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3688075, INGRAM MICR...",D,A,D,PROVEEDOR #,2,1,FDF3688075,INGRAM MICRO SA DE CV


In [ ]:
factura_empresa_list = list(zip(pol_cat['Factura'], pol_cat['Empresa']))

# Display the first few elements of the list
print(factura_empresa_list[:10])

[('B1190023669T10', None), ('FDF3684614', 'INGRAM MICRO SA DE CV'), ('DFA424799', 'CT INTERNACIONAL DEL NOROESTE SA DE CV'), ('DFA425973', 'CT INTERNACIONAL DEL NOROESTE SA DE CV'), ('FDF3688075', 'INGRAM MICRO SA DE CV'), ('AI255038', 'COMPUTACION ADMINISTRATIVA Y DISEÑO SA DE CV'), ('AI255419', 'COMPUTACION ADMINISTRATIVA Y DISEÑO SA DE CV'), ('BPCUAUW14298', 'ESTACION DE SERVICIO CUAUTLA SA DE CV'), ('BKGMEXAACP42327', 'OPERADORA DE FRANQUICIAS ALSEA SAPI DE CV'), ('K701408', 'ASPEL DE MEXICO SA DE CV')]


In [ ]:
pol_cat

,TIPO_POLI,PERIODO,EJERCICIO,NUM_CTA,FECHA_POL,CONCEP_PO,DEBE_HABER,STATUS,TIPO,NOMBRE,NIVEL,NATURALEZA,Factura,Empresa
0,Dr,2,2022,610003900000000000002,2022-02-28 00:00:00.000,PROVISION DE FACTURA NO. B1190023669T10,D,A,D,TELEFONOS,2,0,B1190023669T10,None
1,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3684614, INGRAM MICR...",D,A,D,PROVEEDOR #,2,1,FDF3684614,INGRAM MICRO SA DE CV
2,Eg,2,2022,211000100000000000002,2022-02-03 00:00:00.000,"PAGO DE LA FACTURA NO. DFA424799, CT INTERNACI...",D,A,D,PROVEEDOR #,2,1,DFA424799,CT INTERNACIONAL DEL NOROESTE SA DE CV
3,Eg,2,2022,211000100000000000002,2022-02-08 00:00:00.000,"PAGO DE LA FACTURA NO. DFA425973, CT INTERNACI...",D,A,D,PROVEEDOR #,2,1,DFA425973,CT INTERNACIONAL DEL NOROESTE SA DE CV
4,Eg,2,2022,211000100000000000002,2022-02-10 00:00:00.000,"PAGO DE LA FACTURA NO. FDF3688075, INGRAM MICR...",D,A,D,PROVEEDOR #,2,1,FDF3688075,INGRAM MICRO SA DE CV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2569,Eg,1,2026,610005200000000000002,2026-01-26 00:00:00.000,"PAGO FACTURA NO.AID5F029-...-...5EEF, JUAN FRA...",D,A,D,COMIDAS CON EL PERSONAL,2,0,AID5F029-,JUAN FRANCCISCO LOPEZ VELAZQUEZ
2570,Eg,1,2026,610005200000000000002,2026-01-26 00:00:00.000,"PAGO FACTURA NO.2F5BB468-...-..506F, JUAN FRAN...",D,A,D,COMIDAS CON EL PERSONAL,2,0,2F5BB468-,JUAN FRANCISCO LOPEZ VELAZQUEZ
2571,Eg,1,2026,211000100000000000002,2026-01-27 00:00:00.000,"FACTURA NO.DFC-098948, CT INTERNACIONAL DEL NO...",D,A,D,PROVEEDOR #,2,1,DFC-098948,CT INTERNACIONAL DEL NOROESTE
2572,Eg,1,2026,211000100000000000002,2026-01-29 00:00:00.000,"FACTURA NO.FEH-229195, TONIVISA HOLDING",D,A,D,PROVEEDOR #,2,1,FEH-229195,TONIVISA HOLDING


Con esto podemos unir mediante las facturas que estan registradas con las que aparecen 'xmls'